# Data cleaning and preprocessing

This notebook prepares the client and property datasets for clustering analysis by performing data quality checks, cleaning, feature transformation, and dataset integration.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

clients = pd.read_csv("../data/raw/clients.csv")
properties = pd.read_csv("../data/raw/properties.csv")

print(clients.shape)
print(properties.shape)

(2000, 12)
(10000, 9)


In [2]:
clients = clients.drop_duplicates()
properties = properties.drop_duplicates()

In [3]:
def missing_report(df):
    report = pd.DataFrame({
        "Missing Values": df.isnull().sum(),
        "Percentage": (df.isnull().sum() / len(df)) * 100
    })
    return report.sort_values("Percentage", ascending=False)

missing_report(clients)

,Missing Values,Percentage
client_id,0,0.0
client_type,0,0.0
first_name,0,0.0
last_name,0,0.0
date_of_birth,0,0.0
gender,0,0.0
country,0,0.0
region,0,0.0
acquisition_purpose,0,0.0
satisfaction_score,0,0.0


In [4]:
categorical_cols = [
    "client_type",
    "gender",
    "country",
    "region",
    "acquisition_purpose",
    "loan_applied",
    "referral_channel"
]

for col in categorical_cols:
    clients[col] = (
        clients[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

In [5]:
clients["date_of_birth"] = pd.to_datetime(
    clients["date_of_birth"],
    errors="coerce"
)

current_year = datetime.now().year

clients["age"] = current_year - clients["date_of_birth"].dt.year

In [6]:
clients = clients[(clients["age"] >= 18) & (clients["age"] <= 100)]

In [7]:
buyer_master = properties.merge(
    clients,
    left_on="client_ref",
    right_on="client_id",
    how="left"
)

buyer_master.shape

(10000, 22)

In [8]:
buyer_master.to_csv(
    "../data/processed/buyer_master_dataset.csv",
    index=False
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.
